In [2]:
## Libraries
import numpy as np
import pandas as pd
import librosa
from pathlib import Path
from tqdm import tqdm
from scipy.fftpack import dct

In [3]:
## Sampling rate
sr=16000
## No of frequency bins
n_fft=512
hop_length=160
win_length=320
## No of bark features we want to compress our model into
n_bark=22

In [4]:
def bark_filterbank(sr, n_fft, n_bark=22):
    ## np.linspace(0,x,y) gives linear spacing between the frequencies from 0 to x
    ## here sr/2 taken since the nyquist rate states that max freq *2 is sr
    ## n_fft//2+1 is used because fft returns both positive and negative frequencies
    freqs=np.linspace(0,sr/2,n_fft//2+1)

    ## This is the bark formula
    ## Gives the bark values
    bark=6*np.arcsinh(freqs/600)
    ## Now we scale it accross the frequencies 
    ## here we are binning the frequencies
    bark_bins=np.linspace(bark.min(),bark.max(),n_bark+2)

    ## Creating a 2d matrix of bark_bins times no_of_frequencies
    fb=np.zeros((n_bark, len(freqs)))

    ## Take i in range of no of the barks we want to make
    for i in range(n_bark):
        ## We are taking three variables
        lower=bark_bins[i]
        center=bark_bins[i+1]
        upper=bark_bins[i+2]

        ## For index j and the bark value b for each frequency in bark
        for j,b in enumerate(bark):
            ## If the value of our frequency's bark is bigger than equal to the lower point on scale and smaller than equal to center point on our scale
            if lower<=b<=center:
                
                fb[i,j]=(b-lower)/(center-lower)
            
            elif center<b<=upper:
                fb[i,j]=(upper-b)/(upper-center)

    return fb

bark_fb=bark_filterbank(sr, n_fft)

In [5]:
def extract_rnnoise_features(noisy_audio):
    ## STFT
    stft=librosa.stft(noisy_audio,n_fft=n_fft,hop_length=hop_length,win_length=win_length,window='hann')

    magnitude=np.abs(stft)


    ## Bark Band Energies
    bark_energy=np.dot(bark_fb,magnitude)
    ## log_bark gives human loudness perception
    log_bark=np.log1p(bark_energy)

    ## BFCC
    ## discrete cosine transformation
    ## compresses the loudnes into compact form
    bfcc=dct(log_bark, type=2, axis=0, norm='ortho')
    bfcc=bfcc[:22]
    bfcc=bfcc.T

    ## Delta features
    ## Delts feature measures the change with the next sequence
    delta=librosa.feature.delta(bfcc,order=1)
    delta_delta=librosa.feature.delta(bfcc, order=2)
    delta=delta[:,:6]
    delta_delta=delta_delta[:,:6]


    ## Pitch Features
    ## how high or low does sound feel, thin or heavy
    f0=librosa.yin(noisy_audio,fmin=100,fmax=400,sr=sr,frame_length=win_length,hop_length=hop_length)
    pitch_period=sr/(f0+1e-6)
    pitch_period=pitch_period.reshape(-1,1)

    ## Pitch correlation (simplified)
    ## Finding the correlation in the pitch
    pitch_corr=np.zeros((len(f0),6))
    for i in range(6):
        pitch_corr[:,i]=np.roll(f0, i+1)


    ## Non stationarity
    ## This is the attention method
    ## This finds out the value when something suddenly starts
    ## like 'k','t','p'
    onset=librosa.onset.onset_strength(y=noisy_audio,sr=sr,hop_length=hop_length)
    onset=onset[:len(bfcc)]
    onset=onset.reshape(-1,1)

    ## Final feature vector
    rn_features=np.concatenate([bfcc,delta,delta_delta,pitch_corr,pitch_period,onset],axis=1)
    return rn_features

In [6]:
def compute_rnnoise_gain(clean_audio, noisy_audio, bark_fb=bark_fb, n_fft=512, hop_length=256, win_length=512):
    min_len=min(len(clean_audio),len(noisy_audio))
    clean_audio=clean_audio[:min_len]
    noisy_audio=noisy_audio[:min_len]

    ## STFT
    clean_stft=librosa.stft(clean_audio,n_fft=n_fft,hop_length=hop_length,win_length=win_length,window='hann')

    noisy_stft=librosa.stft(noisy_audio,n_fft=n_fft,hop_length=hop_length,win_length=win_length,window='hann')

    ## Magnitude
    clean_mag=np.abs(clean_stft)
    noisy_mag=np.abs(noisy_stft)

    ## Bark band energy
    clean_bark=np.dot(bark_fb,clean_mag)
    noisy_bark=np.dot(bark_fb,noisy_mag)

    ## Ideal Gain Target
    gain=clean_bark/(noisy_bark+1e-8)

    ## Limit gain range
    gain=np.clip(gain,0,1)

    ## Convert to frame major format
    gain=gain.T
    return gain


In [7]:
clean_folder='16k-LP7'
noisy_folder='noisy_dataset'

clean_files=list(Path(clean_folder).rglob('*.wav'))
noisy_files=list(Path(noisy_folder).rglob('*.wav'))


In [8]:
## Stacking frames
def stack_frames(X,context=2):
    num_frames,num_features=X.shape
    padded=np.pad(X,((context,context),(0,0)),mode='edge')

    stacked=np.zeros((num_frames,num_features*(2*context+1)))

    for i in range(num_frames):
        stacked[i]=padded[i:i+2*context+1].reshape(-1)

    return stacked

In [9]:
'''## Mean and std calculation
all_features=[]
for noisy_path in noisy_files:
    noisy_audio,_=librosa.load(noisy_path,sr=sr)
    features=extract_rnnoise_features(noisy_audio)
    features=stack_frames(features,context=2)
    all_features.append(features)

## Merge everything
all_features=np.vstack(all_features)

## Compute stats
mean=np.mean(all_features,axis=0)
std=np.std(all_features,axis=0)

np.save('mean.npy',mean)
np.save('std.npy',std)
print('Mean, Std computed')'''

"## Mean and std calculation\nall_features=[]\nfor noisy_path in noisy_files:\n    noisy_audio,_=librosa.load(noisy_path,sr=sr)\n    features=extract_rnnoise_features(noisy_audio)\n    features=stack_frames(features,context=2)\n    all_features.append(features)\n\n## Merge everything\nall_features=np.vstack(all_features)\n\n## Compute stats\nmean=np.mean(all_features,axis=0)\nstd=np.std(all_features,axis=0)\n\nnp.save('mean.npy',mean)\nnp.save('std.npy',std)\nprint('Mean, Std computed')"

In [10]:
mean=np.load('mean.npy')
std=np.load('std.npy')

In [11]:
## Sequence overlapping
## Sequence length here is a hyper parameter
def create_overlapping_sequences(features,gain_targets,seq_len=50,stride=25):
    X,Y=[],[]

    total_frames=len(features)

    for start in range(0,total_frames-seq_len, stride):
        end=start+seq_len

        X.append(features[start:end])
        Y.append(gain_targets[start:end])

    return np.array(X,dtype=np.float32),np.array(Y,dtype=np.float32)


In [12]:
## Hyper parameter 50,100,75
SEQ_LEN=50
## Dictionary for fast cleanup
clean_dict={}

In [13]:
def data_generator(noisy_files=noisy_files,clean_dict=clean_dict, mean=mean, std=std, batch_size=16, SEQ_LEN=SEQ_LEN):
    while True:
        print('Generator Running...')
        for file in clean_files:
            clean_dict[file.stem]=file

        X=[]
        Y=[]

        

        for noisy_path in tqdm(noisy_files):
            noisy_name=noisy_path.stem

            clean_key=noisy_name.split('_snr')[0]

            if clean_key not in clean_dict:
                continue

            clean_path=clean_dict[clean_key]

            ## Load audio
            clean_audio,_=librosa.load(clean_path,sr=sr)
            noise_audio,_=librosa.load(noisy_path,sr=sr)

            ## Feature extraction
            features=extract_rnnoise_features(noisy_audio=noise_audio)

            ## Target generation
            gain_targets=compute_rnnoise_gain(clean_audio=clean_audio,noisy_audio=noise_audio)

            ## Align frames
            min_frames=min(len(features),len(gain_targets))
            features=features[:min_frames]
            gain_targets=gain_targets[:min_frames]

            features=stack_frames(features, context=2)

            ## Normalization
            features=(features-mean)/std

            ## dtype conversion
            features=features.astype(np.float32)
            gain_targets=gain_targets.astype(np.float32)
            
            ## Create sequences(per file)
            X_seq,Y_seq=create_overlapping_sequences(features=features,gain_targets=gain_targets)
            
            for x,y in zip(X_seq,Y_seq):
                X.append(x)
                Y.append(y)
                if len(X)==batch_size:
                    yield np.array(X), np.array(Y)
                    X,Y=[],[]

    


In [15]:
## GRU
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GRU, Dense, TimeDistributed,Concatenate,BatchNormalization, Activation, Conv1D, Add


In [16]:

def build_gru_model(seq_len=50, feature_dim=210, output_dim=22):

    ## Input
    inputs=Input(shape=(seq_len,feature_dim))

    x=Conv1D(64, 3,padding='same')(inputs)
    x=BatchNormalization()(x)
    x=Activation('relu')(x)

    x=Conv1D(64, 3,padding='same')(x)

    skip=Dense(64)(inputs)
    x=Add()([x,skip])
    x=Activation('relu')(x)

    ## GRU Layers
    x=GRU(128, return_sequences=True)(x)
    x=GRU(64,return_sequences=True)(x)
    x=Dense(64,activation='relu')(x)

    outputs=Dense(output_dim,activation='sigmoid')(x)

    ## Model
    model=Model(inputs, outputs)
    return model



In [17]:
## Build model
model=build_gru_model()
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 50, 210)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 50, 64)    │     40,384 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 50, 64)    │        256 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 50, 64)    │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 50, 64)    │     12,352 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 50, 64)    │     13,504 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 50, 64)    │          0 │ conv1d_1[0][0],   │
│                     │                   │            │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 50, 64)    │          0 │ add[0][0]         │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru (GRU)           │ (None, 50, 128)   │     74,496 │ activation_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_1 (GRU)         │ (None, 50, 64)    │     37,248 │ gru[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 50, 64)    │      4,160 │ gru_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 50, 22)    │      1,430 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 183,830 (718.09 KB)

 Trainable params: 183,702 (717.59 KB)

 Non-trainable params: 128 (512.00 B)

In [18]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),loss='mse',metrics=['mae','accuracy'])

In [19]:
model.fit(data_generator(noisy_files,clean_dict,mean,std,batch_size=32,SEQ_LEN=50),steps_per_epoch=44,epochs=10)


Generator Running...


  0%|          | 0/1446 [00:00<?, ?it/s]C:\Users\hp\AppData\Local\Temp\ipykernel_9064\2774492702.py:30: UserWarning: With fmin=100.000, sr=16000 and frame_length=320, less than two periods of fmin fit into the frame, which can cause inaccurate pitch detection. Consider increasing to fmin=100.000 or frame_length=321.
  f0=librosa.yin(noisy_audio,fmin=100,fmax=400,sr=sr,frame_length=win_length,hop_length=hop_length)
  1%|          | 14/1446 [00:05<04:26,  5.37it/s] 

Epoch 1/10
 1/44 ━━━━━━━━━━━━━━━━━━━━ 4:31 6s/step - accuracy: 0.0475 - loss: 0.0912 - mae: 0.2490

  1%|▏         | 19/1446 [00:11<15:11,  1.57it/s]

 2/44 ━━━━━━━━━━━━━━━━━━━━ 3s 74ms/step - accuracy: 0.0561 - loss: 0.0961 - mae: 0.2580

...

 20%|█▉        | 285/1446 [00:11<00:41, 28.20it/s]

43/44 ━━━━━━━━━━━━━━━━━━━━ 0s 296ms/step - accuracy: 0.0552 - loss: 0.0887 - mae: 0.2613

 20%|██        | 294/1446 [00:11<00:55, 20.73it/s]

44/44 ━━━━━━━━━━━━━━━━━━━━ 13s 299ms/step - accuracy: 0.0538 - loss: 0.0975 - mae: 0.2810


 21%|██        | 298/1446 [00:11<00:46, 24.51it/s]

In [20]:
def predict_with_chunks(features, model, seq_len=50):
    outputs=[]
    for i in range(0, len(features)-seq_len+1,seq_len):
        chunk=features[i:i+seq_len]
        chunk=np.expand_dims(chunk, axis=0)
        pred=model.predict(chunk, verbose=0)[0]
        outputs.append(pred)
    return np.vstack(outputs)

In [21]:
def pad_features(features, seq_len=50):
    T=len(features)
    remainder=T% seq_len

    if remainder!=0:
        pad_len=seq_len-remainder
        pad=np.zeros((pad_len,features.shape[1]))

        features=np.vstack([features,pad])
    return features,T

In [22]:
## Prediction
def enhance_audio(noisy_audio, model, bark_fb, mean=mean, std=std, n_fft=512, hop_length=hop_length, win_length=win_length):
    ## STFT
    noisy_stft=librosa.stft(noisy_audio, n_fft=n_fft, hop_length=hop_length, window='hann')

    mag=np.abs(noisy_stft)
    phase=np.angle(noisy_stft)


    ## Feature extraction
    features=extract_rnnoise_features(noisy_audio)
    features=stack_frames(features, context=2)

    ## Normalize
    features=(features-mean)/std
    features,original_len=pad_features(features,50)
    pred_gain=predict_with_chunks(features, model, seq_len=50)
    pred_gain=np.sqrt(pred_gain[:original_len])
    ## Convert bark->fft
    fft_gain=np.dot(bark_fb.T, pred_gain.T)

    ## Match shape
    fft_gain=fft_gain[:, :mag.shape[1]]

    ## Apply gain
    enhanced_mag=mag*fft_gain

    ## Reconstruct complex stft
    enhanced_stft=enhanced_mag*np.exp(1j*phase)

    ## Inverse stft
    enhanced_audio=librosa.istft(enhanced_stft,hop_length=hop_length,win_length=win_length,window='hann')

    return enhanced_audio

In [23]:
noisy_files[2]

WindowsPath('noisy_dataset/FG40_04_snr10_noisefreesound_community-industrial-blower-recording-31979.wav')

In [24]:
'noisy_dataset\ML72_10_snr5_noise1-56907-A-46.wav'

'noisy_dataset\\ML72_10_snr5_noise1-56907-A-46.wav'

In [32]:
noisy_audio,sr=librosa.load('noisy_dataset\ML72_09_snr0_noisefreesound_community-snowblower2-88648.wav',sr=sr)



In [33]:
cleaned=enhance_audio(noisy_audio, model, bark_fb)



C:\Users\hp\AppData\Local\Temp\ipykernel_9064\2774492702.py:30: UserWarning: With fmin=100.000, sr=16000 and frame_length=320, less than two periods of fmin fit into the frame, which can cause inaccurate pitch detection. Consider increasing to fmin=100.000 or frame_length=321.
  f0=librosa.yin(noisy_audio,fmin=100,fmax=400,sr=sr,frame_length=win_length,hop_length=hop_length)


In [34]:
import soundfile as sf
sf.write('enhanced1.wav',cleaned,sr)